# An atlas of healthy and injured cell states and niches in the human kidney

Lake et al., Nature 2023 — 论文图表复现

> **说明**：所有代码集中在本 notebook，按模块组织。在 Colab 中 **Runtime → Run all** 即可顺序执行。

| 模块 | 内容 |
|------|------|
| 0 | 环境配置与依赖安装 |
| 1 | 路径与常量配置 |
| 2 | 绘图工具函数 |
| 3 | 数据加载 |
| 4 | Figure 1 — UMAP / 细胞类型 |
| 5+ | 后续 Figure（待补充） |

---
## 模块 0 — 环境配置

In [ ]:
# 安装依赖
!pip install -q scanpy>=1.10.0 anndata>=0.10.0 leidenalg>=0.10.0 python-igraph>=0.11.0 matplotlib>=3.8.0 seaborn>=0.13.0

In [ ]:
# 可选：挂载 Google Drive 存放大型 h5ad 数据
from google.colab import drive

drive.mount("/content/drive")

# 下载数据后在此填写路径，例如：
# H5AD_PATH = "/content/drive/MyDrive/kidney-atlas/kidney.h5ad"
H5AD_PATH = None

---
## 模块 1 — 路径与常量配置

In [ ]:
from pathlib import Path

# 项目根目录（Colab 从 GitHub 打开时自动定位）
PROJECT_ROOT = Path("/content/An-atlas-of-healthy-and-injured-cell-states-and-niches-in-the-human-kidney")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path(".")

FIGURES_DIR = PROJECT_ROOT / "figures"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

for d in [FIGURES_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAPER_DOI = "10.1038/s41586-023-05769-3"
ORIGINAL_CODE_REPO = "https://github.com/KPMP/Cell-State-Atlas-2022"

print("Project root:", PROJECT_ROOT.resolve())

---
## 模块 2 — 绘图工具函数

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc


def setup_scanpy():
    """统一 scanpy / matplotlib 默认设置。"""
    sc.settings.figdir = str(FIGURES_DIR)
    sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
    plt.rcParams["figure.figsize"] = (6, 5)
    plt.rcParams["font.size"] = 10


def save_current_figure(name: str, dpi: int = 300) -> Path:
    """保存当前 matplotlib 图到 figures/ 目录。"""
    out = FIGURES_DIR / name
    plt.savefig(out, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("Saved:", out)
    return out


setup_scanpy()

---
## 模块 3 — 数据加载

数据来源（任选其一）：
- [KPMP Atlas](https://www.kpmp.org/doi-collection/10-48698-3z31-8924)
- [CZ CELLxGENE](https://cellxgene.cziscience.com/)
- [CAP](https://celltype.info/project/588)

In [ ]:
from pathlib import Path

# 优先使用模块 0 中设置的 H5AD_PATH，否则使用默认路径
if H5AD_PATH is None:
    H5AD_PATH = RAW_DATA_DIR / "kidney.h5ad"
else:
    H5AD_PATH = Path(H5AD_PATH)

if not H5AD_PATH.exists():
    raise FileNotFoundError(
        f"未找到数据: {H5AD_PATH}\n"
        "请从 KPMP / CELLxGENE 下载 h5ad，在模块 0 设置 H5AD_PATH 或放到 data/raw/kidney.h5ad"
    )

adata = sc.read_h5ad(H5AD_PATH)
print(adata)
adata

In [ ]:
# 缓存加载结果，避免重复读取大文件
cached_path = PROCESSED_DATA_DIR / "kidney_loaded.h5ad"
adata.write_h5ad(cached_path)
print("Cached:", cached_path)

---
## 模块 4 — Figure 1：UMAP / 细胞类型

根据论文 Figure 1 逐步调整：降维参数、配色、细胞类型注释列名等。

In [ ]:
adata = sc.read_h5ad(PROCESSED_DATA_DIR / "kidney_loaded.h5ad")

# 自动选择细胞类型注释列
color_key = "cell_type" if "cell_type" in adata.obs.columns else adata.obs.columns[0]
print("Coloring by:", color_key)

In [ ]:
if "X_umap" not in adata.obsm:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.pp.pca(adata)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)

sc.pl.umap(adata, color=color_key, save="_figure1_umap.png", show=True)
print("Figure saved to:", FIGURES_DIR / "umap_figure1_umap.png")

---
## 模块 5+ — 后续 Figure（待补充）

在此下方继续添加 Figure 2、3… 的分析与绘图代码。